In [0]:
# =============================================================
# CELL 1 — Config
# =============================================================
GEMINI_API_KEY = "your_gemini_api_key"
GROQ_API_KEY   = "your_groq_api_key"
 
# Qdrant connection (from VPS)
QDRANT_URL     = "your_qdrant_url"  # or http://VPS_IP:6333
 
# Embedding model
EMBED_URL = (
    "https://generativelanguage.googleapis.com/v1beta/models/"
    "gemini-embedding-2:embedContent"
)
 
print("Config loaded.")

Config loaded.


In [0]:
# =============================================================
# CELL 2 — Test dataset
# Create test questions + expected answers (ground truth)
# This will be reference to measure accuration
# =============================================================
 
test_dataset = [
    {
        "question": "How many annual leave days do employees get?",
        "ground_truth": "Employees are entitled to annual leave as specified in their employment agreement."
    },
    {
        "question": "What is the company's policy on remote work?",
        "ground_truth": "Employees may work remotely subject to manager approval and must follow the remote work policy guidelines."
    },
    {
        "question": "What are the IT security requirements for passwords?",
        "ground_truth": "Passwords must be strong, unique, and changed regularly. Employees must not share passwords with anyone."
    },
    {
        "question": "What is the dress code policy?",
        "ground_truth": "Employees are expected to dress professionally and appropriately for their role and work environment."
    },
    {
        "question": "How should data breaches be reported?",
        "ground_truth": "Data breaches must be reported immediately to the IT department and management."
    },
]
 
print(f"Test dataset: {len(test_dataset)} questions loaded.")

Test dataset: 5 questions loaded.


In [0]:
# =============================================================
# CELL 3 — Helper: get embedding
# ================================================m=============
import requests
import time
 
def get_embedding(text: str) -> list:
    resp = requests.post(
        f"{EMBED_URL}?key={GEMINI_API_KEY}",
        json={
            "model": "models/gemini-embedding-2",
            "content": {"parts": [{"text": text}]},
            "outputDimensionality": 768
        },
        timeout=30,
    )
    if resp.status_code == 200:
        return resp.json()["embedding"]["values"]
    raise Exception(f"Embedding error {resp.status_code}: {resp.text}")

In [0]:
# =============================================================
# CELL 4 — Helper: retrieve chunks from Qdrant
# =============================================================
def retrieve_chunks(question: str, top_k: int = 8) -> list:
    vector = get_embedding(question)
    time.sleep(0.1)
 
    resp = requests.post(
        f"{QDRANT_URL}/collections/policy_chunks/points/search",
        headers={"Content-Type": "application/json"},
        json={
            "vector": vector,
            "top": top_k,
            "with_payload": True,
            "score_threshold": 0.3
        },
        timeout=30,
    )
    if resp.status_code != 200:
        raise Exception(f"Qdrant error {resp.status_code}: {resp.text}")
 
    results = resp.json().get("result", [])
    return [
        {
            "text"       : r["payload"]["text"],
            "source_file": r["payload"]["source_file"],
            "topic"      : r["payload"]["topic"],
            "page"       : r["payload"]["page"],
            "score"      : r["score"]
        }
        for r in results
    ]

In [0]:
# =============================================================
# CELL 5 — Helper: generate answer via Groq
# =============================================================
def generate_answer(question: str, chunks: list) -> str:
    if not chunks:
        return "I could not find relevant information in the policy documents."

    context = "\n\n---\n\n".join([
        f"[{i+1}] Source: {c['topic']} ({c['source_file']}, page {c['page']})\n{c['text']}"
        for i, c in enumerate(chunks)
    ])

    prompt = f"""You are a helpful HR assistant for company policies.
Answer the employee's question based ONLY on the policy documents below.
If the answer is not in the documents, say you could not find it.
Always mention which policy document your answer comes from.
Keep your answer concise. Use plain text only.

POLICY DOCUMENTS:
{context}

EMPLOYEE QUESTION: {question}

ANSWER:"""

    for attempt in range(3):
        try:
            resp = requests.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers={
                    "Content-Type": "application/json",
                    "Authorization": f"Bearer {GROQ_API_KEY}"
                },
                json={
                    "model": "llama-3.1-8b-instant",
                    "messages": [{"role": "user", "content": prompt}],
                    "temperature": 0.2,
                    "max_tokens": 512
                },
                timeout=30,
            )
            if resp.status_code == 200:
                return resp.json()["choices"][0]["message"]["content"]
            elif resp.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"  Rate limit, waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Error {resp.status_code}: {resp.text[:100]}")
                break
        except Exception as e:
            print(f"  Exception: {e}")
            break

    return "Could not generate answer after retries."

In [0]:
# =============================================================
# CELL 6 — Run pipeline on test dataset
# Generate answers for all test questions
# =============================================================
import json
 
print("Running RAG pipeline on test dataset...\n")
 
results = []
for i, item in enumerate(test_dataset):
    print(f"[{i+1}/{len(test_dataset)}] {item['question'][:60]}...")
    try:
        chunks  = retrieve_chunks(item["question"])
        answer  = generate_answer(item["question"], chunks)
        context = [c["text"] for c in chunks]
 
        results.append({
            "question"    : item["question"],
            "ground_truth": item["ground_truth"],
            "answer"      : answer,
            "contexts"    : context,
            "chunks_found": len(chunks),
            "top_score"   : chunks[0]["score"] if chunks else 0,
        })
        time.sleep(1)  # avoid rate limit
    except Exception as e:
        print(f"  ERROR: {e}")
 
print(f"\nDone. {len(results)}/{len(test_dataset)} questions processed.")

Running RAG pipeline on test dataset...

[1/5] How many annual leave days do employees get?...
[2/5] What is the company's policy on remote work?...
[3/5] What are the IT security requirements for passwords?...
  Rate limit, waiting 10s...
[4/5] What is the dress code policy?...
  Rate limit, waiting 10s...
  Rate limit, waiting 20s...
[5/5] How should data breaches be reported?...
  Rate limit, waiting 10s...

Done. 5/5 questions processed.


In [0]:
# =============================================================
# CELL 7 — LLM-as-Judge Evaluation
# Ask Gemini to score each answer 1-5 with reasoning
# More practical — no ground truth needed for new questions
# =============================================================
import re
 
def llm_judge(question: str, answer: str, context: list) -> dict:
    context_text = "\n\n".join(context[:3]) if context else "No context retrieved."

    judge_prompt = f"""You are an expert evaluator for a company policy Q&A system.
Evaluate the following answer based on these criteria:
1. Accuracy — Is the answer factually correct based on the context?
2. Completeness — Does the answer fully address the question?
3. Groundedness — Is the answer supported by the provided context?
4. Clarity — Is the answer clear and easy to understand?

QUESTION: {question}

CONTEXT:
{context_text}

ANSWER TO EVALUATE:
{answer}

Respond in this exact JSON format only, no other text:
{{
  "score": <integer 1-5>,
  "accuracy": <integer 1-5>,
  "completeness": <integer 1-5>,
  "groundedness": <integer 1-5>,
  "clarity": <integer 1-5>,
  "reasoning": "<one sentence explanation>"
}}"""

    resp = requests.post(
        f"https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {GROQ_API_KEY}"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": judge_prompt}],
            "temperature": 0.1,
            "max_tokens": 256
        },
        timeout=30,
    )

    # Debug
    print(f"  Status: {resp.status_code}")
    
    if resp.status_code != 200:
        print(f"  Error: {resp.text}")
        return {"score": 0, "reasoning": f"API error: {resp.status_code}"}

    text = resp.json()["choices"][0]["message"]["content"]
    print(f"  Raw response: {text[:100]}")

    try:
        json_match = re.search(r'\{.*\}', text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except Exception as e:
        print(f"  Parse error: {e}")

    return {"score": 0, "reasoning": "Failed to parse judge response"}
 
print("Running LLM-as-Judge evaluation...\n")
 
judge_results = []
for i, r in enumerate(results):
    print(f"[{i+1}/{len(results)}] Judging: {r['question'][:50]}...")
    score = llm_judge(r["question"], r["answer"], r["contexts"])
    judge_results.append({**r, **score})
    time.sleep(1)
 

Running LLM-as-Judge evaluation...

[1/5] Judging: How many annual leave days do employees get?...
  Status: 200
  Raw response: {
  "score": 4,
  "accuracy": 5,
  "completeness": 4,
  "groundedness": 5,
  "clarity": 5,
  "reason
[2/5] Judging: What is the company's policy on remote work?...
  Status: 200
  Raw response: {
  "score": 4,
  "accuracy": 5,
  "completeness": 4,
  "groundedness": 5,
  "clarity": 5,
  "reason
[3/5] Judging: What are the IT security requirements for password...
  Status: 200
  Raw response: {
  "score": 5,
  "accuracy": 5,
  "completeness": 5,
  "groundedness": 5,
  "clarity": 5,
  "reason
[4/5] Judging: What is the dress code policy?...
  Status: 200
  Raw response: {
  "score": 5,
  "accuracy": 5,
  "completeness": 5,
  "groundedness": 5,
  "clarity": 5,
  "reason
[5/5] Judging: How should data breaches be reported?...
  Status: 200
  Raw response: {
  "score": 4,
  "accuracy": 5,
  "completeness": 4,
  "groundedness": 5,
  "clarity": 5,
  "reason


In [0]:
# =============================================================
# CELL 8 — Display LLM-as-Judge results
# =============================================================
print("\n" + "="*70)
print("LLM-AS-JUDGE EVALUATION RESULTS")
print("="*70)
 
total_score = 0
for i, r in enumerate(judge_results):
    print(f"\n[Q{i+1}] {r['question']}")
    print(f"  Score       : {r.get('score', 'N/A')}/5")
    print(f"  Accuracy    : {r.get('accuracy', 'N/A')}/5")
    print(f"  Completeness: {r.get('completeness', 'N/A')}/5")
    print(f"  Groundedness: {r.get('groundedness', 'N/A')}/5")
    print(f"  Clarity     : {r.get('clarity', 'N/A')}/5")
    print(f"  Reasoning   : {r.get('reasoning', 'N/A')}")
    total_score += r.get('score', 0)
 
avg_score = total_score / len(judge_results)
accuracy_pct = (avg_score / 5) * 100
 
print("\n" + "="*70)
print(f"OVERALL RESULTS")
print(f"  Average score   : {avg_score:.2f}/5")
print(f"  Accuracy        : {accuracy_pct:.1f}%")
print(f"  Questions tested: {len(judge_results)}")
print("="*70)


LLM-AS-JUDGE EVALUATION RESULTS

[Q1] How many annual leave days do employees get?
  Score       : 4/5
  Accuracy    : 5/5
  Completeness: 4/5
  Groundedness: 5/5
  Clarity     : 5/5
  Reasoning   : The answer is accurate and grounded in the provided context, but it lacks completeness as it does not provide the actual number of annual leave days, only stating that it is specified in the contract and applicable law.

[Q2] What is the company's policy on remote work?
  Score       : 4/5
  Accuracy    : 5/5
  Completeness: 4/5
  Groundedness: 5/5
  Clarity     : 5/5
  Reasoning   : The answer accurately and clearly states the purpose of the Remote Work Policy, but lacks details on the specifics of the policy, such as eligibility, remote work arrangements, and attendance standards.

[Q3] What are the IT security requirements for passwords?
  Score       : 5/5
  Accuracy    : 5/5
  Completeness: 5/5
  Groundedness: 5/5
  Clarity     : 5/5
  Reasoning   : The answer accurately and completel

In [0]:
# =============================================================
# CELL 9 — Save evaluation results to Delta table
# =============================================================

from pyspark.sql import Row
from datetime import date

spark.sql("""
    CREATE TABLE IF NOT EXISTS policy_eval_results (
        eval_date         DATE,
        eval_type         STRING,
        question          STRING,
        answer            STRING,
        overall_score     DOUBLE,
        accuracy          DOUBLE,
        completeness      DOUBLE,
        groundedness      DOUBLE,
        clarity           DOUBLE,
        reasoning         STRING,
        chunks_found      INT,
        top_score         DOUBLE
    )
    USING DELTA
""")

today = date.today()

rows = []
for r in judge_results:
    rows.append(Row(
        eval_date     = today,
        eval_type     = "llm_as_judge",
        question      = str(r.get("question", "")),
        answer        = str(r.get("answer", "")),
        overall_score = float(r.get("score", 0) or 0),
        accuracy      = float(r.get("accuracy", 0) or 0),
        completeness  = float(r.get("completeness", 0) or 0),
        groundedness  = float(r.get("groundedness", 0) or 0),
        clarity       = float(r.get("clarity", 0) or 0),
        reasoning     = str(r.get("reasoning", "")),
        chunks_found  = int(r.get("chunks_found", 0) or 0),
        top_score     = float(r.get("top_score", 0) or 0),
    ))

from pyspark.sql.types import *
schema = StructType([
    StructField("eval_date",     DateType(),   True),
    StructField("eval_type",     StringType(), True),
    StructField("question",      StringType(), True),
    StructField("answer",        StringType(), True),
    StructField("overall_score", DoubleType(), True),
    StructField("accuracy",      DoubleType(), True),
    StructField("completeness",  DoubleType(), True),
    StructField("groundedness",  DoubleType(), True),
    StructField("clarity",       DoubleType(), True),
    StructField("reasoning",     StringType(), True),
    StructField("chunks_found",  IntegerType(),True),
    StructField("top_score",     DoubleType(), True),
])

df = spark.createDataFrame(rows, schema=schema)
df.write.format("delta").mode("overwrite").saveAsTable("policy_eval_results")

print(f"Saved {len(rows)} evaluation results to Delta table: policy_eval_results")

Saved 5 evaluation results to Delta table: policy_eval_results


In [0]:
# =============================================================
# CELL 10 — Evaluation trend over time
# Run this after multiple evaluation sessions
# =============================================================
spark.sql("""
    SELECT
        eval_date,
        ROUND(AVG(overall_score), 2)        AS avg_score,
        ROUND(AVG(overall_score)/5*100, 1)  AS accuracy_pct,
        ROUND(AVG(accuracy), 2)             AS avg_accuracy,
        ROUND(AVG(completeness), 2)         AS avg_completeness,
        ROUND(AVG(groundedness), 2)         AS avg_groundedness,
        ROUND(AVG(clarity), 2)              AS avg_clarity,
        COUNT(*)                            AS questions_tested
    FROM policy_eval_results
    GROUP BY eval_date
    ORDER BY eval_date DESC
""").show(truncate=False)

+----------+---------+------------+------------+----------------+----------------+-----------+----------------+
|eval_date |avg_score|accuracy_pct|avg_accuracy|avg_completeness|avg_groundedness|avg_clarity|questions_tested|
+----------+---------+------------+------------+----------------+----------------+-----------+----------------+
|2026-06-12|4.4      |88.0        |5.0         |4.4             |5.0             |5.0        |5               |
+----------+---------+------------+------------+----------------+----------------+-----------+----------------+

